Importe a  base de dados do Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Tellus"
AMOSTRA= f"{BASE}/amostras710.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Instale o PyCaret

OBS -> Mudem  o ambiente de excução para o python de 2025/07

Reiniciem a sessão dps de baixar

In [1]:
pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pyod to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.2 MB/s eta 0:00:00


Primeiro passo é passar a sua base para o pandas

Depois, analise  a base, para saber o q precisa ser ajustado

In [2]:
import pandas as pd

df = pd.read_csv(AMOSTRA)

print("--- Primeiras 5 linhas ---")
display(df.head())

print("\n--- Últimas 5 linhas ---")
display(df.tail())

--- Primeiras 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
0,AC-1200013-0065DD410D8B4D74884D124552595A75,-10.065699,-67.099232,165.81,1778.0,25.7,4.8,86.0,PVAd,39.0,2.50,84.8,17.1,Média,23.7,Buriti
1,AC-1200013-008511C82A7D491C8FE9A77F156515BC,-9.766742,-67.135746,140.45,1778.0,25.7,4.8,92.0,PVAd,32.0,2.83,84.8,17.0,Média,480.2,Banana
2,AC-1200013-012F582E360D4D95B8EE515A7BFF367D,-9.917798,-66.662608,131.79,1778.0,25.7,4.9,61.0,PVAd,30.0,4.82,84.8,17.0,Média,48.9,Guaraná
3,AC-1200013-01A560FBDDFF49BBA34A961576C7159C,-9.864668,-66.766198,148.72,1778.0,25.7,4.8,81.0,PVAd,35.0,6.60,84.8,17.0,Média,449.7,Cará
4,AC-1200013-01B3514760CF46D28DD72FF7B6E01D3E,-9.697995,-67.129506,130.97,1829.0,25.7,4.8,82.0,PVAd,33.0,2.26,85.1,17.0,Média,264.1,Banana



--- Últimas 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
704,TO-1700251-0F5AE91371F04AF89749E3D821E339E0,-9.629756,-49.150933,233.64,1617.0,27.3,5.1,69.0,FFc,40.0,2.80,69.2,19.4,Muito alta,182.5,Gengibre
705,TO-1700251-101E7B097D97443AA266549325DB1741,-9.517708,-49.226682,217.25,1617.0,27.3,4.9,73.0,GXbd,40.0,3.73,69.2,19.4,Média,147.8,Gengibre
706,TO-1700251-1029A87BD8D64488AB19BDD20B6A246C,-9.695426,-49.118411,265.17,1617.0,27.3,5.1,66.0,FFc,50.0,2.74,69.2,19.4,Média,392.5,Arroz (sequeiro)
707,TO-1700251-1040BF4854394B6A8A525AAD131B32A5,-9.619685,-49.187555,233.99,1617.0,27.3,5.1,70.0,FFc,38.0,3.20,69.2,19.4,Média,93.9,Gengibre
708,TO-1700251-1041DFD7B0C541B19D05B8098B664C79,-9.695648,-49.220533,233.58,1617.0,27.3,5.0,73.0,FFc,39.0,2.35,69.2,19.4,Média,301.0,Arroz (sequeiro)


Aqui eu defino quem é o X (feature) e quem é o y (target)



In [3]:
FEATURES = [
    "altitude_m",
    "precipitacao_anual_mm",
    "temperatura_media_c",
    "ph_solo",
    "ctc_solo",
    "legenda_solo",
    "mos_solo",
    "declividade_perc",
    "umidade_relativa_perc",
    "radiacao_solar_mj",
    "erosao_classe",
    "distancia_agua_m",
]

TARGET = "cultura_ideal_rotulo"

Aqui eu  converti minhas duas features que estavam em texto para números

In [4]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()

le_solo = LabelEncoder()
le_erosao = LabelEncoder()

df_ml["legenda_solo"] = le_solo.fit_transform(df_ml["legenda_solo"].astype(str))

df_ml["erosao_classe"] = le_erosao.fit_transform(df_ml["erosao_classe"].astype(str))

mapa_solo = dict(zip(le_solo.classes_, le_solo.transform(le_solo.classes_)))
mapa_erosao = dict(zip(le_erosao.classes_, le_erosao.transform(le_erosao.classes_)))

print("\n--- Mapeamento: Legenda Solo ---")
for texto, num in list(mapa_solo.items())[:5]:
    print(f"{num}: {texto}")
print("...")

print("\n--- Mapeamento: Erosão Classe ---")
for texto, num in mapa_erosao.items():
    print(f"{num}: {texto}")

print("\n--- Resumo Final ---")
print("Amostras:", len(df_ml))
print("Classes de cultura mantidas em texto:", df_ml[TARGET].nunique())


--- Mapeamento: Legenda Solo ---
0: AGUA
1: CXbd
2: FFc
3: FXd
4: GXbd
...

--- Mapeamento: Erosão Classe ---
0: Alta
1: Baixa
2: Muito alta
3: Muito baixa
4: Média
5: Área urbana

--- Resumo Final ---
Amostras: 709
Classes de cultura mantidas em texto: 78


Aqui p visualizar a base após a mudança

In [5]:
print("--- Primeiras 5 linhas ---")
display(df_ml.head())

print("\n--- Últimas 5 linhas ---")
display(df_ml.tail())

--- Primeiras 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
0,AC-1200013-0065DD410D8B4D74884D124552595A75,-10.065699,-67.099232,165.81,1778.0,25.7,4.8,86.0,12,39.0,2.50,84.8,17.1,4,23.7,Buriti
1,AC-1200013-008511C82A7D491C8FE9A77F156515BC,-9.766742,-67.135746,140.45,1778.0,25.7,4.8,92.0,12,32.0,2.83,84.8,17.0,4,480.2,Banana
2,AC-1200013-012F582E360D4D95B8EE515A7BFF367D,-9.917798,-66.662608,131.79,1778.0,25.7,4.9,61.0,12,30.0,4.82,84.8,17.0,4,48.9,Guaraná
3,AC-1200013-01A560FBDDFF49BBA34A961576C7159C,-9.864668,-66.766198,148.72,1778.0,25.7,4.8,81.0,12,35.0,6.60,84.8,17.0,4,449.7,Cará
4,AC-1200013-01B3514760CF46D28DD72FF7B6E01D3E,-9.697995,-67.129506,130.97,1829.0,25.7,4.8,82.0,12,33.0,2.26,85.1,17.0,4,264.1,Banana



--- Últimas 5 linhas ---


,cod_imovel,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_c,ph_solo,ctc_solo,legenda_solo,mos_solo,declividade_perc,umidade_relativa_perc,radiacao_solar_mj,erosao_classe,distancia_agua_m,cultura_ideal_rotulo
704,TO-1700251-0F5AE91371F04AF89749E3D821E339E0,-9.629756,-49.150933,233.64,1617.0,27.3,5.1,69.0,2,40.0,2.80,69.2,19.4,2,182.5,Gengibre
705,TO-1700251-101E7B097D97443AA266549325DB1741,-9.517708,-49.226682,217.25,1617.0,27.3,4.9,73.0,4,40.0,3.73,69.2,19.4,4,147.8,Gengibre
706,TO-1700251-1029A87BD8D64488AB19BDD20B6A246C,-9.695426,-49.118411,265.17,1617.0,27.3,5.1,66.0,2,50.0,2.74,69.2,19.4,4,392.5,Arroz (sequeiro)
707,TO-1700251-1040BF4854394B6A8A525AAD131B32A5,-9.619685,-49.187555,233.99,1617.0,27.3,5.1,70.0,2,38.0,3.20,69.2,19.4,4,93.9,Gengibre
708,TO-1700251-1041DFD7B0C541B19D05B8098B664C79,-9.695648,-49.220533,233.58,1617.0,27.3,5.0,73.0,2,39.0,2.35,69.2,19.4,4,301.0,Arroz (sequeiro)


Esse momento é crucial para otimização, eu vi quantas amostras de cada cultura eu tinha, como podem ver, eu tinha varias com somente uma amostra,  e isso quebra o pycaret, já que ele tenta sempre pegar  uma quantidade p teste e outra p treino, e somente com uma amostra n tem como

In [6]:
print(df_ml[TARGET].value_counts())

cultura_ideal_rotulo
Mandioca                     46
Cúrcuma (açafrão)            44
Gergelim                     41
Arroz (sequeiro)             36
Banana                       35
                             ..
Girassol                      1
Milho (2ª safra/safrinha)     1
Orégano                       1
Cana-forrageira               1
Quiabo                        1
Name: count, Length: 78, dtype: int64


Então eu simplesmente peguei e apaguei essas culturas com somente uma amostra

In [7]:
contagem_classes = df_ml[TARGET].value_counts()
classes_validas = contagem_classes[contagem_classes >= 2].index
df_ml = df_ml[df_ml[TARGET].isin(classes_validas)]

ver após tirar os de uma amostra só

In [9]:
print(df_ml[TARGET].value_counts())

cultura_ideal_rotulo
Mandioca                    46
Cúrcuma (açafrão)           44
Gergelim                    41
Arroz (sequeiro)            36
Banana                      35
Gengibre                    30
Agrião                      27
Inhame/Taro                 27
Pinus                       22
Brachiaria/Urochloa         22
Café Arábica                16
Eucalipto                   15
Milho-verde                 14
Uva mesa                    13
Batata-doce                 13
Cará                        12
Sorgo granífero             12
Açaí                        12
Sisal (agave)               11
Hortelã                     11
Melancia                    10
Pequi                       10
Palmito (juçara/pupunha)    10
Muruci/Murici               10
Uva vinho                   10
Café Conilon                 9
Menta                        9
Tungue                       9
Milho (1ª safra)             9
Arroz irrigado (várzea)      8
Vinagreira (Hibisco)         7
Lichia            

Só aqui o pycaret entra

Documentação caso queiram: https://pycaret.readthedocs.io/en/latest/index.html

In [8]:
import pycaret
from pycaret.classification import setup, models,compare_models, pull

setup(
    data=df_ml[FEATURES + [TARGET]],
    target=TARGET,
    session_id=42,
    train_size=0.8,
    fold=5,
    n_jobs=-1,
    use_gpu=False,
    verbose=False,
    fix_imbalance=False,
    normalize=True,
)

best_models = compare_models(
    sort="F1",
    n_select=len(models()),
    turbo=False,
)

leaderboard = pull()

pd.set_option('display.max_rows', None)

display(leaderboard)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.5745,0.0000,0.5745,0.5228,0.5284,0.5594,0.5617,0.3360
xgboost,Extreme Gradient Boosting,0.5620,0.0000,0.5620,0.5233,0.5260,0.5471,0.5489,1.1200
lightgbm,Light Gradient Boosting Machine,0.5602,0.0000,0.5602,0.5181,0.5217,0.5452,0.5472,5.9100
et,Extra Trees Classifier,0.5386,0.0000,0.5386,0.5116,0.5037,0.5228,0.5247,0.3980
mlp,MLP Classifier,0.5242,0.0000,0.5242,0.5043,0.4951,0.5081,0.5099,1.5400
dt,Decision Tree Classifier,0.4776,0.0000,0.4776,0.4682,0.4519,0.4609,0.4629,0.0460
gpc,Gaussian Process Classifier,0.4955,0.0000,0.4955,0.4438,0.4466,0.4773,0.4797,8.0760
gbc,Gradient Boosting Classifier,0.4866,0.0000,0.4866,0.4398,0.4443,0.4677,0.4705,13.0820
knn,K Neighbors Classifier,0.4255,0.0000,0.4255,0.3692,0.3782,0.4049,0.4069,0.0660
lr,Logistic Regression,0.4076,0.0000,0.4076,0.3219,0.3444,0.3839,0.3869,2.1560


Processing:   0%|          | 0/94 [00:00<?, ?it/s]

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.5745,0.0,0.5745,0.5228,0.5284,0.5594,0.5617,0.336
xgboost,Extreme Gradient Boosting,0.5620,0.0,0.5620,0.5233,0.5260,0.5471,0.5489,1.120
lightgbm,Light Gradient Boosting Machine,0.5602,0.0,0.5602,0.5181,0.5217,0.5452,0.5472,5.910
et,Extra Trees Classifier,0.5386,0.0,0.5386,0.5116,0.5037,0.5228,0.5247,0.398
mlp,MLP Classifier,0.5242,0.0,0.5242,0.5043,0.4951,0.5081,0.5099,1.540
dt,Decision Tree Classifier,0.4776,0.0,0.4776,0.4682,0.4519,0.4609,0.4629,0.046
gpc,Gaussian Process Classifier,0.4955,0.0,0.4955,0.4438,0.4466,0.4773,0.4797,8.076
gbc,Gradient Boosting Classifier,0.4866,0.0,0.4866,0.4398,0.4443,0.4677,0.4705,13.082
knn,K Neighbors Classifier,0.4255,0.0,0.4255,0.3692,0.3782,0.4049,0.4069,0.066
lr,Logistic Regression,0.4076,0.0,0.4076,0.3219,0.3444,0.3839,0.3869,2.156
